In [ ]:
Scenario: The product team needs to segment customers based on their purchasing behavior for a new feature rollout.

Database Schema:

user_activity
 user_id
last_login_date
feature_usage_count
account_type
transactions
 transaction_id
user_id
transaction_date
amount
platform
user_preferences
 user_id
communication_preference
interface_theme
notification_settings
Task: Create a SQL query to identify:

Active users (logged in last 30 days)
Filter by high-value customers (top 20% by spending)
User preference trends for the identified customers

In [ ]:
Here’s a clean SQL query that identifies:

✅ **Active users** (logged in within last 30 days)
✅ **High-value customers** (top 20% by total spending)
✅ **User preference trends** for those customers

```sql
WITH active_users AS (
    -- Step 1: Get active users
    SELECT
        user_id,
        account_type,
        feature_usage_count
    FROM user_activity
    WHERE last_login_date >= CURRENT_DATE - INTERVAL '30 days'
),

customer_spending AS (
    -- Step 2: Calculate total spending
    SELECT
        user_id,
        SUM(amount) AS total_spent
    FROM transactions
    GROUP BY user_id
),

high_value_customers AS (
    -- Step 3: Select top 20% spenders
    SELECT
        user_id,
        total_spent
    FROM (
        SELECT
            user_id,
            total_spent,
            NTILE(5) OVER (
                ORDER BY total_spent DESC
            ) AS spending_group
        FROM customer_spending
    ) ranked
    WHERE spending_group = 1
),

identified_customers AS (
    -- Step 4: Combine active + high-value
    SELECT
        au.user_id,
        au.account_type,
        au.feature_usage_count,
        hv.total_spent
    FROM active_users au
    INNER JOIN high_value_customers hv
        ON au.user_id = hv.user_id
)

-- Step 5: Preference trends
SELECT
    up.communication_preference,
    up.interface_theme,
    up.notification_settings,
    COUNT(DISTINCT ic.user_id) AS customer_count,
    AVG(ic.feature_usage_count) AS avg_feature_usage,
    AVG(ic.total_spent) AS avg_spending
FROM identified_customers ic
INNER JOIN user_preferences up
    ON ic.user_id = up.user_id
GROUP BY
    up.communication_preference,
    up.interface_theme,
    up.notification_settings
ORDER BY customer_count DESC;
```

### How it works

1. **`active_users`**

   * Filters users who logged in within the **last 30 days**.

2. **`customer_spending`**

   * Calculates total spending per user.

3. **`high_value_customers`**

   * Uses `NTILE(5)` to divide users into **5 spending groups**.
   * Group **1 = top 20% spenders**.

4. **`identified_customers`**

   * Keeps only users who are both:

     * Active
     * High-value

5. **Final Query**

   * Joins with preferences to reveal **behavior trends**, such as:

     * Preferred communication method
     * Popular interface theme
     * Notification settings
   * Also shows:

     * Average spending
     * Average feature usage

### Example Insight

You may discover:

| Theme      | Communication | Customers |
| ---------- | ------------: | --------: |
| Dark Mode  |         Email |       250 |
| Light Mode |           SMS |        80 |

This helps the product team decide:

* Which feature experience to prioritize
* Best communication channel for rollout
* Which UI theme high-value users prefer
